# llminfer Serving Notebook (Colab)

OpenAI-compatible local API in Colab, with:
- robust startup check,
- non-streaming chat completion,
- SSE streaming parser,
- simple concurrency load test + plot,
- health + metrics checks.



## 0) Runtime
In Colab, choose **Runtime -> Change runtime type -> GPU** before running.


In [ ]:
!nvidia-smi

import os
import platform
import torch

print('Python :', platform.python_version())
print('Torch  :', torch.__version__)
print('CUDA   :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU    :', torch.cuda.get_device_name(0))
else:
    print('Warning: CUDA is not enabled. Use a GPU runtime for realistic results.')


## 1) Clone + install serving extras

In [ ]:
REPO_URL = 'https://github.com/nickforce989/llminfer.git'
TARGET_DIR = '/content/llminfer'

import os
import shutil

if os.path.exists(TARGET_DIR):
    shutil.rmtree(TARGET_DIR)

!git clone -q {REPO_URL} {TARGET_DIR}
%cd {TARGET_DIR}

!pip -q install -U pip
!pip -q install -e ".[serve,peft]"


## 2) Start server process

In [ ]:
import os
import signal
import subprocess
import time
import requests

SERVER_HOST = '127.0.0.1'
SERVER_PORT = 8000
MODEL = 'facebook/opt-125m'

if 'server_proc' in globals() and server_proc and server_proc.poll() is None:
    os.kill(server_proc.pid, signal.SIGTERM)
    time.sleep(1)

cmd = [
    'python', '-m', 'llminfer.cli', 'serve',
    '--model', MODEL,
    '--backend', 'eager',
    '--host', SERVER_HOST,
    '--port', str(SERVER_PORT),
    '--max-batch-size', '16',
    '--batch-timeout-ms', '20',
]

server_proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

deadline = time.time() + 180
ready = False
while time.time() < deadline:
    if server_proc.poll() is not None:
        raise RuntimeError('Server exited early. Check logs in the next cell.')
    try:
        r = requests.get(f'http://{SERVER_HOST}:{SERVER_PORT}/healthz', timeout=2)
        if r.ok:
            ready = True
            print('Server ready:', r.json())
            break
    except Exception:
        pass
    time.sleep(1)

if not ready:
    raise RuntimeError('Server did not become ready within timeout')


## 3) Optional: inspect recent server logs

In [ ]:
import itertools

if server_proc.stdout is None:
    print('No stdout pipe')
else:
    print('--- recent logs (up to 40 lines) ---')
    lines = []
    for _ in range(40):
        line = server_proc.stdout.readline()
        if not line:
            break
        lines.append(line.rstrip())
    for ln in lines[-40:]:
        print(ln)


## 4) Non-streaming chat completion

In [ ]:
import requests

payload = {
    'model': 'local-llminfer',
    'messages': [
        {'role': 'system', 'content': 'You are concise.'},
        {'role': 'user', 'content': 'Tell me about GPUs in 4 bullet points.'},
    ],
    'max_tokens': 160,
    'temperature': 0.2,
}

resp = requests.post(
    f'http://{SERVER_HOST}:{SERVER_PORT}/v1/chat/completions',
    json=payload,
    timeout=180,
)
resp.raise_for_status()
data = resp.json()

print('Response text:\n')
print(data['choices'][0]['message']['content'])
print('\nUsage:', data.get('usage'))


## 5) Streaming chat completion (robust SSE parser)

In [ ]:
import json
import requests

stream_payload = {
    'model': 'local-llminfer',
    'messages': [{'role': 'user', 'content': 'Explain KV cache in under 80 words.'}],
    'max_tokens': 120,
    'temperature': 0.2,
    'stream': True,
}

decoder = json.JSONDecoder()
finished = False

with requests.post(
    f'http://{SERVER_HOST}:{SERVER_PORT}/v1/chat/completions',
    json=stream_payload,
    stream=True,
    timeout=180,
) as r:
    r.raise_for_status()

    for raw_line in r.iter_lines(decode_unicode=True):
        if not raw_line:
            continue
        line = raw_line.strip()
        if 'data:' not in line:
            continue

        payloads = [p.strip() for p in line.split('data:') if p.strip()]
        for payload in payloads:
            if payload == '[DONE]':
                finished = True
                break
            if payload in {'\\n', '\\n\\n', '"\\n"', '"\\n\\n"'}:
                continue

            s = payload
            while s:
                s = s.lstrip()
                if not s:
                    break
                try:
                    obj, idx = decoder.raw_decode(s)
                except json.JSONDecodeError:
                    break
                s = s[idx:]
                delta = obj.get('choices', [{}])[0].get('delta', {}).get('content', '')
                if delta:
                    print(delta, end='', flush=True)

        if finished:
            print('\n\n[done]')
            break


## 6) Concurrency smoke test + latency plot

In [ ]:
import concurrent.futures as cf
import statistics
import time
import matplotlib.pyplot as plt
import requests

N_REQ = 12
MAX_WORKERS = 4


def call_once(i: int):
    p = {
        'model': 'local-llminfer',
        'messages': [{'role': 'user', 'content': f'Give one short fact about GPUs #{i}.'}],
        'max_tokens': 64,
        'temperature': 0.2,
    }
    t0 = time.perf_counter()
    r = requests.post(f'http://{SERVER_HOST}:{SERVER_PORT}/v1/chat/completions', json=p, timeout=180)
    r.raise_for_status()
    dt = (time.perf_counter() - t0) * 1000
    txt = r.json()['choices'][0]['message']['content']
    return dt, txt

latencies = []
with cf.ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futs = [ex.submit(call_once, i) for i in range(N_REQ)]
    for fut in cf.as_completed(futs):
        dt, txt = fut.result()
        latencies.append(dt)

print(f'Requests: {len(latencies)}')
print(f'Latency mean: {statistics.mean(latencies):.1f} ms')
print(f'Latency p95 : {sorted(latencies)[int(0.95 * (len(latencies)-1))]:.1f} ms')

plt.figure(figsize=(6, 3.6))
plt.hist(latencies, bins=8)
plt.title('Latency Distribution (ms)')
plt.xlabel('Latency (ms)')
plt.ylabel('Count')
plt.grid(alpha=0.3)
plt.show()


## 7) Health + metrics

In [ ]:
import requests

print('Healthz:')
print(requests.get(f'http://{SERVER_HOST}:{SERVER_PORT}/healthz', timeout=10).json())

metrics_resp = requests.get(f'http://{SERVER_HOST}:{SERVER_PORT}/metrics', timeout=10)
print('\nMetrics status:', metrics_resp.status_code)
print('\n'.join(metrics_resp.text.splitlines()[:25]))


## 8) Cleanup

In [ ]:
import os
import signal
import time

if 'server_proc' in globals() and server_proc and server_proc.poll() is None:
    os.kill(server_proc.pid, signal.SIGTERM)
    time.sleep(1)
    print('Server stopped.')
else:
    print('No active server process.')
